# comparison.ipynb: R1, R2, R4, R5, R6, R7
Run top to bottom. Data: `Telco-Customer-Churn.csv` next to this file (else OpenML id 42178). Set env `QUICK=1` only for a smoke test.

In [ ]:
import os, time, platform, pickle, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, log_loss
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV,
                                     RandomizedSearchCV, cross_val_score, cross_validate,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore", category=FutureWarning)   # sklearn >= 1.9 deprecates criterion="friedman_mse" (same as squared_error)
SEED = 42
QUICK = os.environ.get("QUICK") == "1"   # smoke-test mode only; leave unset for the real run

# ---- data: Telco Customer Churn (IBM sample data) ----
if os.path.exists("Telco-Customer-Churn.csv"):
    df = pd.read_csv("Telco-Customer-Churn.csv")
else:
    from sklearn.datasets import fetch_openml
    df = fetch_openml(data_id=42178, as_frame=True).frame
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")
X = df.drop(columns=["customerID", "Churn"], errors="ignore")   # 11 NaNs stay; imputed inside pipelines
y = (df["Churn"].astype(str) == "Yes").astype(int)
num = X.select_dtypes("number").columns.tolist()
cat = X.columns.difference(num).tolist()

# ---- metric (named in R1 before any fit): recall at the top 10% of scores ----
def recall_at_top10(y_true, y_score, frac=0.10):
    y_true = np.asarray(y_true); n = len(y_true); k = int(np.ceil(frac * n))
    rng = np.random.default_rng(0)
    order = np.lexsort((rng.random(n), -np.asarray(y_score)))   # random tie-break
    return y_true[order[:k]].sum() / y_true.sum()

scorer = make_scorer(recall_at_top10, response_method="predict_proba")

def prep(scale):
    steps = [("imp", SimpleImputer(strategy="constant", fill_value=0))]   # no billing yet = 0
    if scale: steps.append(("sc", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(steps), num),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), cat)])

# ---- one held-out test split, untouched until R5 ----
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y, random_state=SEED)
X_tr, X_te, y_tr, y_te = X.iloc[idx_tr], X.iloc[idx_te], y.iloc[idx_tr], y.iloc[idx_te]
print(df.shape, "| train", X_tr.shape, "| test", X_te.shape, "| QUICK =", QUICK)

## R1: data audit, deployment, metric

In [ ]:
y_raw = (df["Churn"] == "Yes").astype(int)
print("target:\n", y_raw.value_counts(), "\nbase rate:", round(y_raw.mean(), 4))
print("\nmissing values:\n", X.isna().sum()[lambda s: s > 0])
print("rows with missing TotalCharges, tenure values:", df.loc[df["TotalCharges"].isna(), "tenure"].unique())
print("\nfeature types:\n", X.dtypes.map(lambda t: "numeric" if t.kind in "if" else "categorical").value_counts())

D = pd.get_dummies(X.fillna({"TotalCharges": 0}), drop_first=True).astype(float)
C = D.corr().abs()
pairs = C.where(np.triu(np.ones(C.shape), 1).astype(bool)).stack().sort_values(ascending=False)
print("\ntop correlated pairs:\n", pairs.head(15).round(3))
top = pairs.index[0][0]
print("\ncolumns with |r| > 0.99 to", top, ":", C.index[C[top] > 0.99].tolist())

**Deployment.** A weekly batch job scores all ~7,000 active customers, so latency is irrelevant. The model is retrained monthly, or sooner if drift monitoring fires. The retention team can contact only ~10% (~700) per week. Account managers and the retention lead must understand why each customer is flagged, so predictions need reason codes.

**Primary metric (named before any fit): recall at the top 10% of predicted churn scores.** Only ~700 customers can be contacted, so what matters is the share of true churners in that slice. Accuracy misleads at a 26.5% base rate; AUC-ROC ignores the capacity cutoff. PR-AUC and calibration are secondary.

## R2: baselines (full data, repeated CV)

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=1 if QUICK else 3, random_state=SEED)
dummy = DummyClassifier(strategy="prior")
logreg = Pipeline([("prep", prep(scale=True)), ("clf", LogisticRegression(max_iter=1000))])
tree = GridSearchCV(
    Pipeline([("prep", prep(scale=False)), ("clf", DecisionTreeClassifier(random_state=SEED))]),
    param_grid={"clf__ccp_alpha": np.logspace(-5, -2, 4 if QUICK else 12)},
    scoring=scorer, cv=StratifiedKFold(5, shuffle=True, random_state=0))

for name, model in [("Dummy (prior)", dummy), ("LogReg (L2, default)", logreg),
                    ("Pruned tree (nested CV)", tree)]:
    s = cross_val_score(model, X, y, cv=cv, scoring=scorer, n_jobs=-1)
    print(f"{name:26s} recall@10% = {s.mean():.3f} ± {s.std():.3f}")
tree.fit(X, y); print("chosen ccp_alpha:", tree.best_params_["clf__ccp_alpha"])

## R4: tuning (training split only)

In [ ]:
# OOB-vs-trees curve -> fixed n_estimators
rf = RandomForestClassifier(max_features="sqrt", min_samples_leaf=5, warm_start=True,
                            oob_score=True, n_jobs=-1, random_state=0)
ns = list(range(100, 301, 100)) if QUICK else list(range(50, 1001, 50))
ll, rec = [], []
A_tr = prep(scale=False).fit(X_tr).transform(X_tr)
for n in ns:
    rf.set_params(n_estimators=n).fit(A_tr, y_tr)
    p = rf.oob_decision_function_[:, 1]
    ll.append(log_loss(y_tr, p)); rec.append(recall_at_top10(y_tr, p))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(ns, ll); ax[0].set(xlabel="trees", ylabel="OOB log-loss")
ax[1].plot(ns, rec); ax[1].set(xlabel="trees", ylabel="OOB recall@10%")
plt.tight_layout(); plt.show()
conv = next(n for i, n in enumerate(ns) if all(abs(l - ll[-1]) < 0.001 for l in ll[i:]))
N_FIXED = int(np.ceil(1.5 * conv / 50) * 50)
print(f"OOB log-loss within 0.001 of its final value from ~{conv} trees -> n_estimators FIXED at {N_FIXED}")

In [ ]:
N_ITER = 3 if QUICK else 30
rf_search = RandomizedSearchCV(
    Pipeline([("prep", prep(scale=False)),
              ("clf", RandomForestClassifier(n_estimators=N_FIXED, n_jobs=1, random_state=0))]),
    param_distributions={"clf__max_features": loguniform(0.03, 0.6),
                         "clf__min_samples_leaf": loguniform(5e-4, 2e-2),
                         "clf__max_samples": uniform(0.4, 0.6)},
    n_iter=N_ITER, cv=cv, scoring=scorer, n_jobs=-1, random_state=0)
gb_search = RandomizedSearchCV(
    Pipeline([("prep", prep(scale=False)),
              ("clf", HistGradientBoostingClassifier(max_iter=1000, early_stopping=True,
                      validation_fraction=0.15, n_iter_no_change=20, random_state=0))]),
    param_distributions={"clf__learning_rate": loguniform(0.01, 0.3),
                         "clf__l2_regularization": loguniform(1e-3, 10),
                         "clf__max_leaf_nodes": randint(4, 48),
                         "clf__min_samples_leaf": [10, 20, 40, 80, 160]},
    n_iter=N_ITER, cv=cv, scoring=scorer, n_jobs=-1, random_state=0)

def report(name, s):
    r = pd.DataFrame(s.cv_results_); b = s.best_index_
    m, sd = r.mean_test_score[b], r.std_test_score[b]
    k = int((r.mean_test_score >= m - sd).sum())
    print(f"{name:14s} best CV recall@10% = {m:.3f} ± {sd:.3f} | {k}/{len(r)} candidates within 1 SD of best")
    print("   best params:", {a.split('__')[1]: (round(float(v), 5) if isinstance(v, (float, np.floating)) else v)
                              for a, v in s.best_params_.items()})

rf_search.fit(X_tr, y_tr); report("Random forest", rf_search)
gb_search.fit(X_tr, y_tr); report("Boosting", gb_search)
print("boosting rounds picked by early stopping (refit on full train):", gb_search.best_estimator_["clf"].n_iter_)

## R5: paired comparison (same repeated folds, training split)

In [ ]:
rf_pipe, gb_pipe = rf_search.best_estimator_, gb_search.best_estimator_   # tuned, refit on train
lin = clone(logreg).fit(X_tr, y_tr)

cv55 = RepeatedStratifiedKFold(n_splits=5, n_repeats=2 if QUICK else 5, random_state=7)  # identical splits for all three
models = {"linear": clone(logreg), "forest": clone(rf_pipe), "boosting": clone(gb_pipe)}
res = {k: cross_validate(m, X_tr, y_tr, cv=cv55, scoring=scorer, n_jobs=-1) for k, m in models.items()}
S = pd.DataFrame({k: v["test_score"] for k, v in res.items()})
print(S.agg(["mean", "std"]).round(3))

MARGIN = 0.02      # smallest recall@10% gain judged worth the extra cost
rows = []
for a, b in [("forest", "linear"), ("boosting", "linear"), ("forest", "boosting")]:
    d = S[a] - S[b]; k = len(d)
    t = d.mean() / np.sqrt((1 / k + 1 / 4) * d.var(ddof=1))       # Nadeau-Bengio, test/train = 1/4
    p = 2 * stats.t.sf(abs(t), df=k - 1)
    rows.append({"pair": f"{a} - {b}", "mean diff": d.mean(), "sd diff": d.std(ddof=1),
                 "wins": f"{(d > 0).sum()}/{k}", "losses": int((d < 0).sum()),
                 "p (corrected)": p, "p (Holm-3)": min(1, p * 3)})
T = pd.DataFrame(rows).round(4)
T["verdict"] = np.where(T["p (Holm-3)"] < 0.05, "clearly different", "not shown to be different")
T["gain > margin?"] = T["mean diff"] > MARGIN
print(T.to_string(index=False))
print(pd.DataFrame({k: {"fit s/fold": v["fit_time"].mean(), "score s/fold": v["score_time"].mean()}
                    for k, v in res.items()}).T.round(3))

# single held-out test split: sanity check only (about 141 customers in the top 10%, so noisy)
print({k: round(recall_at_top10(y_te, m.predict_proba(X_te)[:, 1]), 3)
       for k, m in [("linear", lin), ("forest", rf_pipe), ("boosting", gb_pipe)]})

## R6: reliance and explanation (held-out test split)

In [ ]:
N_REP = 5 if QUICK else 30
pi = {k: permutation_importance(m, X_te, y_te, scoring=scorer, n_repeats=N_REP, random_state=0, n_jobs=-1)
      for k, m in [("forest", rf_pipe), ("boosting", gb_pipe)]}
PI = pd.DataFrame({f"{k}_{s}": getattr(v, f"importances_{s}")
                   for k, v in pi.items() for s in ("mean", "std")}, index=X_te.columns)

names = rf_pipe["prep"].get_feature_names_out()
def orig(n):
    n = n.split("__", 1)[1]
    return max([c for c in X_te.columns if n == c or n.startswith(c + "_")], key=len)
PI["forest_impurity"] = (pd.Series(rf_pipe["clf"].feature_importances_, index=names)
                         .groupby(orig).sum().reindex(X_te.columns))
PI["rank_perm"] = PI["forest_mean"].rank(ascending=False)
PI["rank_imp"] = PI["forest_impurity"].rank(ascending=False)
PI["rank_gap"] = (PI["rank_perm"] - PI["rank_imp"]).abs()
print(PI.sort_values("forest_mean", ascending=False).round(4).to_string())
print("\nrankings that disagree by more than 3 positions (discuss each):")
print(PI.loc[PI["rank_gap"] > 3, ["forest_mean", "forest_impurity", "rank_perm", "rank_imp"]].round(4).to_string())

GROUP = ["InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
         "TechSupport", "StreamingTV", "StreamingMovies"]      # R1 group, original columns
def group_perm(est, X, y, cols, n_repeats=N_REP, seed=0):
    rng, base, out = np.random.default_rng(seed), scorer(est, X, y), []
    for _ in range(n_repeats):
        Xp = X.copy(); Xp[cols] = X[cols].values[rng.permutation(len(X))]   # ONE permutation for the whole group
        out.append(base - scorer(est, Xp, y))
    return np.array(out)
for k, m in [("forest", rf_pipe), ("boosting", gb_pipe)]:
    g = group_perm(m, X_te, y_te, GROUP)
    best = PI.loc[GROUP, f"{k}_mean"].idxmax()
    print(f"{k:9s} GROUP: {g.mean():.4f} ± {g.std():.4f} | largest single column ({best}): "
          f"{PI.loc[best, f'{k}_mean']:.4f} ± {PI.loc[best, f'{k}_std']:.4f}")

In [ ]:
# one test case: three predictions + what each model can say
P = pd.DataFrame({"linear": lin.predict_proba(X_te)[:, 1], "forest": rf_pipe.predict_proba(X_te)[:, 1],
                  "boosting": gb_pipe.predict_proba(X_te)[:, 1]}, index=X_te.index)
cut = int(np.ceil(0.10 * len(P)))
case = P.mean(axis=1).sort_values(ascending=False).index[0]     # change [0] to inspect other customers
print("actual churn:", bool(y.loc[case])); print(P.loc[case].round(3).to_string())
print("in top 10% of test scores:", (P.rank(ascending=False).loc[case] <= cut).to_dict())
print(X_te.loc[case].to_string())
z = lin["prep"].transform(X_te.loc[[case]])
contrib = pd.Series(lin["clf"].coef_[0] * z[0], index=lin["prep"].get_feature_names_out())
print("\ntop linear contributions to the log-odds:")
print(contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(6).round(3).to_string())

## R7: cost, counted

In [ ]:
B  = rf_pipe["prep"].transform(X_te).astype(np.float64)
Bg = gb_pipe["prep"].transform(X_te).astype(np.float64)

rf_trees = [e.tree_ for e in rf_pipe["clf"].estimators_]
rf_stored = sum(t.node_count for t in rf_trees)
ind, _ = rf_pipe["clf"].decision_path(B)
rf_visited = ind.sum() / len(B)

def hgb_walk(tree, Z):
    nd, cur, n = tree.nodes, np.zeros(len(Z), dtype=int), np.ones(len(Z))
    act = np.flatnonzero(nd["is_leaf"][cur] == 0)        # == 0, not ~ (is_leaf is uint8)
    for _ in range(int(nd["depth"].max()) + 2):          # guard: cannot loop forever
        if not act.size: break
        r = nd[cur[act]]
        cur[act] = np.where(Z[act, r["feature_idx"]] <= r["num_threshold"], r["left"], r["right"])
        n[act] += 1
        act = np.flatnonzero(nd["is_leaf"][cur] == 0)
    return cur, n

gb_clf = gb_pipe["clf"]
gb_trees = [p[0] for p in gb_clf._predictors]
gb_stored = sum(len(t.nodes) for t in gb_trees)
leaf_sum, gb_visited = np.zeros(len(Bg)), 0.0
for t in gb_trees:
    cur, n = hgb_walk(t, Bg); leaf_sum += t.nodes["value"][cur]; gb_visited += n.mean()
raw = float(np.ravel(gb_clf._baseline_prediction)[0]) + leaf_sum
print("traversal matches decision_function:", np.allclose(raw, gb_clf.decision_function(Bg)))

Bl = lin["prep"].transform(X_te)
p_coef = lin["clf"].coef_.shape[1]; nnz = (Bl != 0).sum(axis=1).mean()
cost = pd.DataFrame({
    "trees": [len(rf_trees), len(gb_trees), "-"],
    "nodes stored": [rf_stored, gb_stored, f"{p_coef} coef + 1 intercept"],
    "nodes visited / prediction": [round(rf_visited, 1), round(gb_visited, 1), "-"],
    "comparisons / prediction": [round(rf_visited - len(rf_trees), 1), round(gb_visited - len(gb_trees), 1), "-"],
    "arithmetic / prediction": ["+ 1 add per tree", "+ 1 add per tree", f"{p_coef} mult + {p_coef} add + 1 sigmoid"],
    "pickled size (KB)": [len(pickle.dumps(m)) // 1024 for m in (rf_pipe, gb_pipe, lin)],
}, index=["forest", "boosting", "linear"])
print(cost.T.to_string())

In [ ]:
def timeit(f, runs=5):
    t = []
    for _ in range(runs):
        s = time.perf_counter(); f(); t.append(time.perf_counter() - s)
    return f"{np.median(t):.3f}s  ({min(t):.3f}-{max(t):.3f})"
rf1 = clone(rf_pipe).set_params(clf__n_jobs=1); rfN = clone(rf_pipe).set_params(clf__n_jobs=-1)
fit_t = {"linear": timeit(lambda: clone(logreg).fit(X_tr, y_tr)),
         "forest, n_jobs=1": timeit(lambda: clone(rf1).fit(X_tr, y_tr)),
         "forest, n_jobs=-1": timeit(lambda: clone(rfN).fit(X_tr, y_tr)),
         "boosting (early stop)": timeit(lambda: clone(gb_pipe).fit(X_tr, y_tr))}
rf1.fit(X_tr, y_tr)
pred_t = {"linear": timeit(lambda: lin.predict_proba(X_te)), "forest": timeit(lambda: rf1.predict_proba(X_te)),
          "boosting": timeit(lambda: gb_pipe.predict_proba(X_te))}
print(f"FIT (train rows = {len(X_tr)}), median and (min-max) of 5 runs:")
for k, v in fit_t.items(): print(f"  {k:24s} {v}")
print(f"PREDICT (batch of {len(X_te)} rows), median and (min-max) of 5 runs:")
for k, v in pred_t.items(): print(f"  {k:24s} {v}")
print("machine:", platform.platform(), "|", platform.processor() or "cpu n/a",
      "| logical cores:", os.cpu_count(), "| python", platform.python_version())